# TANAGER - Exp001: Cryosphere retrieval, end to end

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/seantangth/tanager-cryo/blob/main/2_notebooks/TANAGER_03_train_exp001_cryo_retrieval_e2e.ipynb)

Purpose: reproduce the whole submission - train the retrieval, evaluate accuracy and calibration, apply it to a real Tanager scene, check it against same-day Sentinel-2, and re-verify the memo's headline numbers.
Runtime: CPU - Colab CPU is sufficient; no GPU needed
Why: the network has 213,644 parameters and 80 epochs of training is ~20 s of compute. The bottleneck is network I/O (an 883 MB scene) and catalogue queries, not arithmetic. Uploading the training set to a rented GPU would take longer than training on it.
Estimated time: ~3 min with `DRY_RUN = True`; ~35 min full, of which ~15 min is downloading
Inputs: Tanager Open STAC (public GCS), Sentinel-2 L2A COGs (public AWS), NASA CMR - **no credentials required anywhere**
Outputs: `4_models/cryo_retrieval.pt`, `5_outputs/*.nc`, `5_outputs/*.json`, `5_outputs/figures/*.png`
Prerequisites: None

---

**The argument this notebook supports.** On 26 August 2026 a rock-and-ice slope failure on Langtang Lirung left more than 600 people confirmed dead and over 1,900 missing (as of 29 August; the toll is still rising). A spaceborne imaging spectrometer had looked at that slope clearly twice in three and a half years, the last time 28 months earlier. At 73 N the count is structurally zero - EMIT flies on the ISS and cannot reach past 52 degrees, and PRISMA stops at 70. And the Tanager Open STAC catalogue contains no glacier at all: all 153 scenes, checked against the OpenStreetMap glacier layer, return zero intersections.

So the retrieval is built and validated on the cryosphere data that *does* exist - Arctic sea ice in melt - and the case is made for pointing Tanager at ice.

In [ ]:
# === Cell 1: Setup & Imports ===
import time

_cell_times = {}
_cell_start = None


def cell_start(name):
    global _cell_start
    _cell_start = time.time()
    _cell_times[name] = None
    print(f'\u25b6 {name}')


def cell_end(name):
    elapsed = time.time() - _cell_start
    _cell_times[name] = elapsed
    print(f'\u2713 {name} \u2014 {elapsed:.1f}s ({elapsed/60:.1f}min)')


cell_start('Cell 1: Setup & Imports')

import json
import os
import subprocess
import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
print('python', sys.version.split()[0])

cell_end('Cell 1: Setup & Imports')

In [ ]:
# === Cell 2: Config ===
cell_start('Cell 2: Config')

# Set False for the full run. True keeps every step but shrinks the expensive ones,
# so the whole notebook still exercises the real code path.
DRY_RUN = True

SEED = 0
SNR_THRESHOLD = 30.0
SCENE_ID = '20250606_181248_58_4001'

N_TRAIN = 8_000 if DRY_RUN else 150_000
N_VAL = 3_000 if DRY_RUN else 30_000
EPOCHS = 8 if DRY_RUN else 80
BATCH_SIZE = 512
LR = 2e-3

# The public repository, so the Colab badge above works with no setup. Override with
# TANAGER_REPO_URL, or export TANAGER_REPO to point at a checkout you already have.
REPO_URL = os.environ.get(
    'TANAGER_REPO_URL', 'https://github.com/seantangth/tanager-cryo.git')

MARKER = Path('3_src') / 'tanager_cryo' / 'model.py'


def locate_repo():
    """Find the repository: an explicit override, an enclosing checkout, or a Colab clone.

    Failing loudly here is deliberate. An earlier version fell through to `git clone` of a
    placeholder URL, which failed and then took fifteen downstream cells with it as
    NameErrors. A single clear message beats a cascade.
    """
    env = os.environ.get('TANAGER_REPO')
    if env and (Path(env) / MARKER).exists():
        return Path(env).resolve()

    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / MARKER).exists():
            return candidate

    in_colab = 'google.colab' in sys.modules
    if in_colab and REPO_URL:
        target = Path('/content') / 'PBC_Tanager_Open_Data'
        if not (target / MARKER).exists():
            print(f'cloning {REPO_URL} ...')
            subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(target)], check=True)
        return target

    raise RuntimeError(
        'Cannot find the repository.\n'
        f'  cwd = {here}\n'
        '  Run this notebook from inside a checkout (2_notebooks/ is the natural place),\n'
        '  or set TANAGER_REPO=/path/to/checkout,\n'
        '  or on Colab set TANAGER_REPO_URL to the public clone URL.'
    )


REPO_ROOT = locate_repo()
SRC = REPO_ROOT / '3_src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

DATA = REPO_ROOT / '1_data'
MODELS = REPO_ROOT / '4_models'
OUT = REPO_ROOT / '5_outputs'
FIGS = OUT / 'figures'
SCENE_PATH = DATA / 'raw' / 'sirmilik' / f'{SCENE_ID}_sr.h5'
# A dry run trains on a fraction of the data, so its outputs must never overwrite the
# verified ones committed to the repository.
SUFFIX = '_dryrun' if DRY_RUN else ''
S2_ITEM = DATA / 'raw' / 's2_item.json'
for d in (MODELS, OUT, FIGS):
    d.mkdir(parents=True, exist_ok=True)

IN_COLAB = 'google.colab' in sys.modules
print(f'repo      {REPO_ROOT}')
print(f'colab     {IN_COLAB}')
print(f'DRY_RUN   {DRY_RUN}  (n_train={N_TRAIN:,}, epochs={EPOCHS})')

cell_end('Cell 2: Config')

In [ ]:
# === Cell 3: Install & Smoke Test ===
cell_start('Cell 3: Install & Smoke Test')

if IN_COLAB:
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO_ROOT / 'requirements.txt')],
        check=True,
    )

import matplotlib.pyplot as plt
import numpy as np
import rasterio
import torch
import xarray as xr

runtime = 'cuda' if torch.cuda.is_available() else (
    'mps' if torch.backends.mps.is_available() else 'cpu')
print(f'torch {torch.__version__}  runtime={runtime}')
print(f'xarray {xr.__version__}  rasterio {rasterio.__version__}')

# Planet's official reader registers itself as an xarray backend engine. If this name is
# missing the scene cannot be opened, so fail loudly here rather than ten cells later.
engines = xr.backends.list_engines()
assert 'tanager' in engines, f"xarray-hyperspectral not installed; engines={list(engines)}"
print("xarray engine 'tanager' registered \u2713")

cell_end('Cell 3: Install & Smoke Test')

## 1. The gap

Everything below is computed from public catalogues at run time, not quoted from the memo.

In [ ]:
# === Cell 4: The observation gap ===
cell_start('Cell 4: The observation gap')

import datetime as dt

from tanager_cryo import observation_gap as ogap

records = ogap.query_point(*ogap.LANGTANG)
summary = ogap.summarise(records, ogap.LANGTANG_FAILURE_DATE)

last = dt.date.fromisoformat(summary['last_usable_before_failure'])
fail = dt.date.fromisoformat(ogap.LANGTANG_FAILURE_DATE)
GAP_MONTHS = (fail.year - last.year) * 12 + fail.month - last.month

print(f"Langtang source zone {ogap.LANGTANG[1]:.4f} N, {ogap.LANGTANG[0]:.4f} E")
print(f"  EMIT L2A granules covering the point : {summary['n_observations']}")
print(f"  usable at < {ogap.USABLE_CLOUD_MAX:.0f}% cloud            : {summary['n_usable']}")
for date, cc in summary['usable_dates']:
    print(f"      {date}   cloud {cc:.0f}%")
print(f"  last usable look before the failure  : {summary['last_usable_before_failure']}")
print(f"  gap                                  : {GAP_MONTHS} months")

# Robustness: two source positions have been published, 1.1 km apart. An EMIT granule is
# ~75 km across, so the headline result should not depend on which one is used. Checked
# rather than asserted.
alt = ogap.summarise(ogap.query_point(*ogap.LANGTANG_ALT), ogap.LANGTANG_FAILURE_DATE)
ALT_SAME = all(summary[k] == alt[k] for k in
               ('n_observations', 'n_usable', 'last_usable_before_failure'))
print(f"  same result at the alternative published position: {ALT_SAME}")

print(f"\nSirmilik 73.72 N : EMIT observations = 0, structurally.")
print('  the ISS orbit reaches only +/- 52 degrees; PRISMA stops at 70')

# Suffixed: a dry run must never overwrite the audit committed to the repository.
json.dump(
    {'langtang': {'site': {'lon': ogap.LANGTANG[0], 'lat': ogap.LANGTANG[1]},
                  'failure_date': ogap.LANGTANG_FAILURE_DATE,
                  'summary': summary, 'records': records,
                  'alt_position': {'lon': ogap.LANGTANG_ALT[0],
                                   'lat': ogap.LANGTANG_ALT[1],
                                   'summary': alt,
                                   'same_headline_result': ALT_SAME}},
     'sirmilik': {'latitude': ogap.SIRMILIK[1],
                  'emit_latitude_limit': ogap.EMIT_LATITUDE_LIMIT,
                  'emit_can_observe': False, 'n_observations': 0,
                  'reason': ('the ISS orbit reaches only about +/- 52 degrees, so EMIT '
                             'never overflies this latitude; PRISMA acquires only within '
                             "+/- 70 degrees, and EnMAP's proposal-driven tasked archive "
                             'holds no systematic sea-ice coverage')}},
    open(OUT / f'observation_gap{SUFFIX}.json', 'w'), indent=1)

cell_end('Cell 4: The observation gap')

In [ ]:
# === Cell 5: Glacier audit ===
cell_start('Cell 5: Glacier audit')

import urllib.request

from tanager_cryo import fetch as tfetch
from tanager_cryo.glacier_check import glacier_count

# The full audit walks all 153 scenes and takes ~10 minutes of Overpass round trips; it
# lives in tanager_cryo.glacier_check and its result is committed to
# 5_outputs/glacier_audit.json. The Snow and Ice collection is the subset most likely to
# contain a glacier, so a dry run checks those seven and reports that it did so.
col = json.load(urllib.request.urlopen(f'{tfetch.CATALOG}/snow-ice/collection.json', timeout=60))
snow_ids = [l['href'].rsplit('/', 1)[-1].removesuffix('.json')
            for l in col['links'] if l['rel'] == 'item']

checked, hits, skipped = 0, 0, 0
for sid in snow_ids:
    item = tfetch.find_item(sid)
    n = glacier_count(item['bbox'])
    if n is None:
        skipped += 1
        continue
    checked += 1
    hits += (n > 0)
    print(f'  {sid}  glaciers={n}')

print(f'\nchecked {checked} Snow and Ice scenes, {hits} intersect a mapped glacier'
      + (f' ({skipped} skipped)' if skipped else ''))

audit = OUT / 'glacier_audit.json'
if audit.exists():
    a = json.load(open(audit))
    print(f"full audit ({a['date']}): {a['n_checked']}/{a['n_unique_scenes']} scenes "
          f"checked, {a['n_intersecting']} intersect a mapped glacier")
if DRY_RUN:
    print('DRY_RUN: the full 153-scene sweep is skipped here - run '
          '`python -m tanager_cryo.glacier_check` to reproduce it')

cell_end('Cell 5: Glacier audit')

## 2. The scene, and the band selection its own uncertainty layer implies

In [ ]:
# === Cell 6: Fetch Tanager scene ===
cell_start('Cell 6: Fetch Tanager scene')

item = tfetch.find_item(SCENE_ID)
print(f"{SCENE_ID}: {item['properties'].get('description', '')[-60:]}")
print(f"  sun elevation {item['properties'].get('view:sun_elevation')}\u00b0  "
      f"off-nadir {item['properties'].get('view:off_nadir')}\u00b0")

if SCENE_PATH.exists():
    print(f'  already present: {SCENE_PATH.stat().st_size / 1e6:.0f} MB')
elif DRY_RUN:
    tfetch.download_asset(item, 'thumbnail', SCENE_PATH.parent, '_thumb.png')
    print('  DRY_RUN: surface-reflectance cube (883 MB) not downloaded')
else:
    tfetch.download_asset(item, 'ortho_sr_hdf5', SCENE_PATH.parent, '_sr.h5')

HAVE_SCENE = SCENE_PATH.exists()
print(f'\nHAVE_SCENE = {HAVE_SCENE}')

cell_end('Cell 6: Fetch Tanager scene')

In [ ]:
# === Cell 7: Band selection from uncertainty ===
cell_start('Cell 7: Band selection from uncertainty')

from tanager_cryo import synth, viz

viz.use_style()

wl_all = np.load(DATA / 'optical_constants' / 'tanager_wavelengths.npy')
snr = np.load(DATA / 'optical_constants' / 'tanager_snr_median.npy')
good = np.load(DATA / 'optical_constants' / 'tanager_good_wavelengths.npy').astype(bool)
wl, sigma, band_index = synth.load_band_selection(SNR_THRESHOLD)

print(f'426 bands \u2192 {good.sum()} pass good_wavelengths '
      f'\u2192 {wl.size} pass SNR > {SNR_THRESHOLD:.0f}')
print(f'retained range {wl.min():.0f}\u2013{wl.max():.0f} nm')

fig, ax = plt.subplots(figsize=(8.6, 3.6))
ax.semilogy(wl_all[good], snr[good], color=viz.SERIES[0], lw=1.6, label='median SNR')
ax.axhline(SNR_THRESHOLD, color=viz.INK_MUTED, ls='--', lw=1.0)
ax.text(2450, SNR_THRESHOLD * 1.15, f'SNR = {SNR_THRESHOLD:.0f}', fontsize=8,
        color=viz.INK_MUTED, ha='right')
ax.scatter(wl, snr[band_index], s=6, color=viz.SERIES[1], zorder=3, label='retained')
ax.set_xlabel('Wavelength (nm)')
ax.set_ylabel('Reflectance / uncertainty')
ax.set_title("Band selection is derived from the scene's own uncertainty layer")
ax.grid(True, which='both')
ax.set_axisbelow(True)
ax.legend(loc='upper right')
plt.show()

N_BANDS = int(wl.size)
cell_end('Cell 7: Band selection from uncertainty')

## 3. Forward model, and the synthetic training set built from it

In [ ]:
# === Cell 8: Forward model sanity ===
cell_start('Cell 8: Forward model sanity')

from tanager_cryo import forward as F

MU0 = 0.624  # cos(solar zenith) for this scene
at = lambda r, t: r[int(np.abs(wl - t).argmin())]

fine = F.snow_reflectance(wl, 1e-4, MU0)
coarse = F.snow_reflectance(wl, 1e-2, MU0)
dirty = F.snow_reflectance(wl, 1e-3, MU0, lap_load=50.0)
clean = F.snow_reflectance(wl, 1e-3, MU0)
shallow = F.melt_pond_reflectance(wl, 0.10, mu0=MU0)
deep = F.melt_pond_reflectance(wl, 0.40, mu0=MU0)
water = F.open_water_reflectance(wl)

checks = [
    ('grain size up -> 1030 nm darkens', at(coarse, 1030) < at(fine, 1030)),
    ('grain size up -> 400 nm barely moves', abs(at(coarse, 400) - at(fine, 400)) < 0.06),
    ('pond deeper -> 660 nm darkens', at(deep, 660) < at(shallow, 660)),
    ('pond deeper -> blue survives better than red',
     (at(shallow, 440) - at(deep, 440)) < (at(shallow, 660) - at(deep, 660))),
    ('impurities darken the visible, not the NIR',
     (at(clean, 400) - at(dirty, 400)) > (at(clean, 1240) - at(dirty, 1240))),
]
for label, ok in checks:
    print(f"  [{'ok  ' if ok else 'FAIL'}] {label}")
assert all(ok for _, ok in checks), 'forward model failed a physical sanity check'

fig, ax = plt.subplots(figsize=(8.6, 3.8))
for spec, lab, c in ((fine, 'fine snow, L=1e-4 m', viz.SERIES[0]),
                     (coarse, 'bare ice, L=1e-2 m', viz.SERIES[1]),
                     (shallow, 'melt pond, 10 cm', viz.SERIES[2])):
    ax.plot(wl, spec, color=c, label=lab)
ax.plot(wl, water, color=viz.INK_MUTED, lw=1.4, ls='--', label='open water')
ax.set_xlabel('Wavelength (nm)')
ax.set_ylabel('Reflectance')
ax.set_title('Endmember spectra from the forward model')
ax.grid(True)
ax.set_axisbelow(True)
ax.legend()
plt.show()

cell_end('Cell 8: Forward model sanity')

In [ ]:
# === Cell 9: Synthetic training set ===
cell_start('Cell 9: Synthetic training set')

train = synth.generate(n=N_TRAIN, mu0=MU0, seed=SEED, snr_threshold=SNR_THRESHOLD)
val = synth.generate(n=N_VAL, mu0=MU0, seed=SEED + 9973, snr_threshold=SNR_THRESHOLD)
print(f'{train.reflectance.shape[0]:,} train / {val.reflectance.shape[0]:,} val, '
      f'{train.wavelength.size} bands')

# The perturbation emulating forward-model error must be smooth. White noise of the same
# amplitude would teach the network to be uncertain rather than robust.
rng = np.random.default_rng(SEED)
pert = synth.smooth_model_error(wl, 4000, rng)
adj = float(np.sqrt((np.diff(pert, axis=1) ** 2).mean()))
white = float(np.sqrt(2) * np.sqrt((pert ** 2).mean()))
print(f'model-error perturbation: RMS {np.sqrt((pert**2).mean()):.4f}, '
      f'band-to-band step {adj:.5f} vs {white:.5f} for white noise '
      f'({white/adj:.0f}x smoother)')

if HAVE_SCENE:
    ds_chk = xr.open_dataset(SCENE_PATH, engine='tanager')
    v = (~ds_chk.beta_cloud_mask.values) & (~ds_chk.nodata_pixels.values)
    ys, xs = np.where(v)
    pick = np.random.default_rng(2).choice(len(ys), min(6000, len(ys)), replace=False)
    obs = ds_chk.reflectance.values[train.band_index][:, ys[pick], xs[pick]].T
    obs = obs[np.isfinite(obs).all(1)]
    lo = np.percentile(train.reflectance, 0.5, axis=0)
    hi = np.percentile(train.reflectance, 99.5, axis=0)
    inside = (obs >= lo) & (obs <= hi)
    print(f'coverage of observed spectra: per-band median '
          f'{np.median(inside.mean(0)) * 100:.1f}%')
    ds_chk.close()

cell_end('Cell 9: Synthetic training set')

In [ ]:
# === Cell 10: Train ===
cell_start('Cell 10: Train')

from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

from tanager_cryo.model import (RetrievalNet, build_features, fit_standardiser,
                                gaussian_nll)

torch.manual_seed(SEED)
device = torch.device(runtime)

x_tr = build_features(train.reflectance, train.uncertainty)
x_va = build_features(val.reflectance, val.uncertainty)
std = fit_standardiser(x_tr, train.params)

to_t = lambda a: torch.from_numpy(a.astype(np.float32))
dl = DataLoader(
    TensorDataset(to_t(std.encode_x(x_tr)),
                  to_t((train.params - std.y_mean) / std.y_std)),
    batch_size=BATCH_SIZE, shuffle=True)
xv = to_t(std.encode_x(x_va)).to(device)
yv = to_t((val.params - std.y_mean) / std.y_std).to(device)

net = RetrievalNet(x_tr.shape[1]).to(device)
opt = torch.optim.AdamW(net.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
print(f'{sum(p.numel() for p in net.parameters()):,} parameters on {device}')

best = float('inf')
bar = tqdm(range(1, EPOCHS + 1), desc='train')
for ep in bar:
    net.train()
    run = 0.0
    for xb, yb in dl:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        mean, logvar = net(xb)
        loss = gaussian_nll(mean, logvar, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(net.parameters(), 5.0)
        opt.step()
        run += loss.item() * xb.shape[0]
    sched.step()
    net.eval()
    with torch.no_grad():
        m, lv = net(xv)
        vnll = gaussian_nll(m, lv, yv).item()
    bar.set_postfix(train=f'{run/len(dl.dataset):.3f}', val=f'{vnll:.3f}')
    if vnll < best:
        best = vnll
        torch.save({'state_dict': net.state_dict(), 'n_features': x_tr.shape[1],
                    'width': 256,
                    'standardiser': {k: v.tolist() for k, v in std.to_dict().items()},
                    'band_index': train.band_index.tolist(),
                    'wavelength': train.wavelength.tolist(),
                    'param_names': list(synth.PARAM_NAMES),
                    'mu0': MU0, 'snr_threshold': SNR_THRESHOLD},
                   MODELS / f'cryo_retrieval{SUFFIX}.pt')

BEST_VAL_NLL = best
print(f'best val NLL {best:.4f} \u2192 {MODELS / f"cryo_retrieval{SUFFIX}.pt"}')

cell_end('Cell 10: Train')

## 4. Accuracy is the easy half. Calibration is the half that matters.

In [ ]:
# === Cell 11: Evaluate ===
cell_start('Cell 11: Evaluate')

from tanager_cryo.evaluate import load_model, predict

net_e, std_e, ckpt = load_model(MODELS / f'cryo_retrieval{SUFFIX}.pt', device)
test = synth.generate(n=N_VAL, mu0=MU0, seed=4242, snr_threshold=SNR_THRESHOLD)
mean, sig = predict(net_e, std_e, device, test.reflectance, test.uncertainty)
names = ckpt['param_names']

print(f"{'parameter':30s} {'RMSE':>9} {'R2':>7}   {'conditional':>30}")
for j, nm in enumerate(names):
    err = mean[:, j] - test.params[:, j]
    r2 = 1.0 - (err ** 2).mean() / test.params[:, j].var()
    cond = ''
    if nm in synth.CONDITIONAL_PARAMS:
        host, thr = synth.CONDITIONAL_PARAMS[nm]
        k = names.index(host)
        m = test.params[:, k] > thr
        e = mean[m, j] - test.params[m, j]
        cond = (f'{host}>{thr}: RMSE {np.sqrt((e**2).mean()):.4f} '
                f'R2 {1 - (e**2).mean() / test.params[m, j].var():5.3f}')
    print(f'{nm:30s} {np.sqrt((err**2).mean()):9.4f} {r2:7.3f}   {cond:>30}')

print(f"\n{'parameter':30s} {'var(z)':>8} {'68% cov':>9} {'95% cov':>9}")
VAR_Z = []
for j, nm in enumerate(names):
    z = (test.params[:, j] - mean[:, j]) / np.clip(sig[:, j], 1e-9, None)
    VAR_Z.append(float(z.var()))
    print(f'{nm:30s} {z.var():8.2f} {(np.abs(z) < 1).mean():9.3f} '
          f'{(np.abs(z) < 1.96).mean():9.3f}')
print('nominal: var(z) = 1.0, coverage 0.683 and 0.950')

cell_end('Cell 11: Evaluate')

## 5. The real scene, including where the model does not fit it

In [ ]:
# === Cell 12: Retrieve on the real scene ===
cell_start('Cell 12: Retrieve on the real scene')

RETRIEVAL_NC = OUT / f'sirmilik_retrieval{SUFFIX}.nc'

if HAVE_SCENE:
    from tanager_cryo.retrieve import retrieve_scene
    result = retrieve_scene(SCENE_PATH, MODELS / f'cryo_retrieval{SUFFIX}.pt')
    result.to_netcdf(RETRIEVAL_NC,
                     encoding={v: {'zlib': True, 'complevel': 5} for v in result.data_vars})
    FWD_ERR = result.attrs['forward_model_error_median']
    INST_SIG = result.attrs['instrument_sigma_median']
    print(f'\nforward-model error {FWD_ERR:.4f}  vs instrument sigma {INST_SIG:.4f}'
          f'  \u2192 forward-model limited by {FWD_ERR / INST_SIG:.0f}x')
elif RETRIEVAL_NC.exists():
    result = xr.open_dataset(RETRIEVAL_NC)
    FWD_ERR = result.attrs['forward_model_error_median']
    INST_SIG = result.attrs['instrument_sigma_median']
    print('using the retrieval shipped with the repository (scene not downloaded)')
else:
    result, FWD_ERR, INST_SIG = None, float('nan'), float('nan')
    print('DRY_RUN without the scene or a stored retrieval: skipping')

cell_end('Cell 12: Retrieve on the real scene')

## 6. The independent check

Sentinel-2C imaged this scene 29 minutes after Tanager, on the same grid, at 10 m. Tanager infers the sub-pixel fractions *spectrally*; Sentinel-2 resolves them *spatially*. An earlier version of this model passed every synthetic test and failed this one - see the memo.

In [ ]:
# === Cell 13: Independent validation ===
cell_start('Cell 13: Independent validation')

VAL_JSON = OUT / f's2_validation{SUFFIX}.json'
DARK_R, DARK_BIAS = float('nan'), float('nan')

if result is not None and S2_ITEM.exists():
    rc = subprocess.run(
        [sys.executable, '-m', 'tanager_cryo.s2_validate',
         '--retrieval', str(RETRIEVAL_NC), '--s2-json', str(S2_ITEM),
         '--out', str(VAL_JSON), '--out-nc', str(OUT / f's2_validation{SUFFIX}.nc')],
        cwd=str(REPO_ROOT), env={**os.environ, 'PYTHONPATH': str(SRC)},
        capture_output=True, text=True)
    print(rc.stdout[-1800:] or rc.stderr[-1200:])

if VAL_JSON.exists():
    dm = json.load(open(VAL_JSON))['metrics_dark_fraction_linear']
    DARK_R, DARK_BIAS = dm['pearson_r'], dm['bias']
    print(f"\ndark-surface fraction: r = {DARK_R:+.3f}, bias {DARK_BIAS:+.3f} "
          f"(Tanager {dm['tanager_mean']:.3f} vs Sentinel-2 {dm['sentinel2_mean']:.3f})")
    print('the offset is a physical limit: at 30 m, passive optics cannot separate open '
          'water from thin dark ice')
else:
    print('validation not run (needs the scene)')

cell_end('Cell 13: Independent validation')

In [ ]:
# === Cell 14: Sensor comparison ===
cell_start('Cell 14: Sensor comparison')

from tanager_cryo import s2compare

CMP_JSON = OUT / f'sensor_comparison{SUFFIX}.json'
rc = subprocess.run(
    [sys.executable, '-m', 'tanager_cryo.experiment_s2',
     '--n', str(N_TRAIN), '--n-val', str(N_VAL),
     '--epochs', str(EPOCHS), '--out', str(CMP_JSON)],
    cwd=str(REPO_ROOT), env={**os.environ, 'PYTHONPATH': str(SRC)},
    capture_output=True, text=True)
print(rc.stdout[-1400:] or rc.stderr[-1200:])

GRAIN_RATIO = float('nan')
if CMP_JSON.exists():
    GRAIN_RATIO = json.load(open(CMP_JSON))['summary'][
        'log10_absorption_length_m']['ratio']
    print(f'\ngrain size costs {GRAIN_RATIO:.1f}x more error on Sentinel-2 bands - it lives '
          'in the ice features at 1030 and 1240 nm, and S2 L2A has nothing between 865 '
          'and 1614 nm')

# The same protocol against the EMIT and PRISMA band sets. Neither sensor can acquire this
# latitude, so degrading Tanager to their published sampling and FWHM is the only way to
# compare them here.
HSI_JSON = OUT / f'sensor_comparison_hsi{SUFFIX}.json'
rc = subprocess.run(
    [sys.executable, '-m', 'tanager_cryo.experiment_hsi',
     '--n', str(N_TRAIN), '--n-val', str(N_VAL),
     '--epochs', str(EPOCHS), '--out', str(HSI_JSON)],
    cwd=str(REPO_ROOT), env={**os.environ, 'PYTHONPATH': str(SRC)},
    capture_output=True, text=True)
print(rc.stdout[-1200:] or rc.stderr[-1200:])

HSI_RANGE = (float('nan'), float('nan'))
if HSI_JSON.exists():
    s = json.load(open(HSI_JSON))['summary']
    ratios = [s[k][f'{sensor}_ratio'] for k in s for sensor in ('emit', 'prisma')]
    HSI_RANGE = (min(ratios), max(ratios))
    print(f'\nEMIT and PRISMA band sets: RMSE ratios {HSI_RANGE[0]:.2f}-{HSI_RANGE[1]:.2f} '
          'across all six parameters - statistically unchanged. The cliff is between '
          'multispectral and imaging spectroscopy, so the Arctic gap is orbital, not '
          'spectral.')

cell_end('Cell 14: Sensor comparison')

In [ ]:
# === Cell 15: Figures ===
cell_start('Cell 15: Figures')

# Figures are rebuilt from the committed outputs, not the dry-run ones.
rc = subprocess.run([sys.executable, '-m', 'tanager_cryo.figures'],
                    cwd=str(REPO_ROOT), env={**os.environ, 'PYTHONPATH': str(SRC)},
                    capture_output=True, text=True)
print(rc.stdout[-900:] or rc.stderr[-900:])

N_FIGS = len(list(FIGS.glob('*.png')))
for p in sorted(FIGS.glob('*.png')):
    print(f'  {p.name}  {p.stat().st_size / 1e3:.0f} kB')

cell_end('Cell 15: Figures')

In [ ]:
# === Cell 16: Verify ===
cell_start('Cell 16: Verify')

rc = subprocess.run([sys.executable, '-m', 'tanager_cryo.verify'],
                    cwd=str(REPO_ROOT), env={**os.environ, 'PYTHONPATH': str(SRC)},
                    capture_output=True, text=True)
print(rc.stdout[-2600:] or rc.stderr[-1200:])
VERIFY_OK = (rc.returncode == 0)
if DRY_RUN:
    print('\nNote: in DRY_RUN this checks the outputs committed to the repository, not the '
          'reduced ones this notebook just produced (those carry a _dryrun suffix so they '
          'cannot overwrite anything). Set DRY_RUN = False to regenerate and re-verify '
          'from scratch.')

cell_end('Cell 16: Verify')

In [ ]:
# === Cell 17: Summary ===
cell_start('Summary')

status = 'ok' if (VERIFY_OK or DRY_RUN) else 'verify_failed'
print(f"""notebook: TANAGER_03_train_exp001_cryo_retrieval_e2e.ipynb
DRY_RUN: {DRY_RUN}
status: {status}
bands: {N_BANDS} retained at SNR>{SNR_THRESHOLD:.0f}
val_nll: {BEST_VAL_NLL:.4f}
calibration: var(z)={min(VAR_Z):.2f}-{max(VAR_Z):.2f} (nominal 1.0)
obs_gap: {GAP_MONTHS} months, {summary['n_usable']}/{summary['n_observations']} EMIT looks usable
obs_gap_robust: same result at the alternative source position = {ALT_SAME}
fwd_vs_instrument: {FWD_ERR:.4f} / {INST_SIG:.4f}
s2_validation: r={DARK_R:+.3f}, bias={DARK_BIAS:+.3f}
grain_size_penalty: {GRAIN_RATIO:.1f}x on Sentinel-2 bands
emit_prisma_penalty: {HSI_RANGE[0]:.2f}-{HSI_RANGE[1]:.2f}x (statistically unchanged)
verify: {('all claims match' + (' (committed outputs)' if DRY_RUN else '')) if VERIFY_OK else 'see Cell 16'}
outputs: {OUT} ({N_FIGS} figures)
""")

cell_end('Summary')